In [24]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import os
from multiprocessing import Pool
from scipy.stats import entropy

In [25]:
accessions=['at6923', 'at6929', 'at7143', 'at8285', 'at9104', 'at9336', 'at9503', 'at9578', 'at9744', 'at9762', 'at9806', 'at9830', 'at9847', 'at9852', 'at9879', 'at9883', 'at9900']

In [27]:
def simpson(frequencies):
    return sum(f**2 for f in frequencies)

In [28]:
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

#for acc in tqdm(accessions):
def get_entropies_per_gene(acc):
    # load sqanti classification file
    at=pd.read_csv(f"output/01/{acc}-hodgepodgemerged_classification_hodgepodge-subset.txt", sep="\t")
    at["cat_subcat"]=at["structural_category"] + "_" + at["subcategory"]
    
    # get associated hodgepodge gene (one per row)
    span = 2
    at["hodgepodge_gene"]=at["associated_gene"].transform(lambda x: ["_".join(x.split("_")[i:i+span]) for i in range(0, len(x.split("_")), span)])
    at_hodge=at.explode("hodgepodge_gene")
    at_hodge=at_hodge.loc[at_hodge["hodgepodge_gene"].str.split("_").str[0]==acc]


    # subset for analysis
    at_info=at_hodge[["chrom","strand","cat_subcat","isoform_reads","hodgepodge_gene", "isoform","ORF_seq"]]
    
    # start analysis per gene
    acc_df=pd.DataFrame()
    for gene in at_info.groupby(by="hodgepodge_gene"):
        gene[1].fillna("noncoding", inplace=True)
        #print(gene[0])
        g_df=pd.DataFrame()
        #get gene and strand
        g_df["hodgepodge_gene"]=[gene[0]]
        g_df["strand"]=gene[1]["strand"].unique()[0]
        g_df["chrom"]=gene[1]["chrom"].unique()[0]
        
        #prepare analyses
        gene_rs=gene[1]["isoform_reads"].sum()
        gene_isoforms=len(gene[1]["isoform"])
        
        g_df["read_support"]=gene_rs
        g_df["isoforms"]=gene_isoforms

        #calculate diversity based on isoforms and their read supports
        isoform_frequencies=[]
        for i in gene[1]["isoform"].unique().tolist():
            #get read support for cat-combination
            rs=gene[1].loc[gene[1]["isoform"]==i]["isoform_reads"].sum() 
            isoform_frequencies.append(rs/gene_rs)
        
        g_df["isoform_shannon"]= entropy(isoform_frequencies)
        g_df["isoform_simpson"]= simpson(isoform_frequencies)
        
        
        ORF_frequencies=[]
        #calculate diversity based on reads, supporting ORFs
        for i in gene[1]["ORF_seq"].unique().tolist():
            #get read support for cat-combination
            rs=gene[1].loc[gene[1]["ORF_seq"]==i]["isoform_reads"].sum() 
            ORF_frequencies.append(rs/gene_rs)
            
        g_df["orf_shannon"]=entropy(ORF_frequencies)
        g_df["orf_simpson"]=simpson(ORF_frequencies)

        #calculate shannon entropy based on isoforms, supporting cat-subcat combinations
        #shannon_iso_cat_subcat=[]
        #for i in gene[1]["cat_subcat"].unique().tolist():
        #    #get n idoforms of for cat-combination
        #    n_iso=len(gene[1].loc[gene[1]["cat_subcat"]==i])  
        #    shannon_iso_cat_subcat.append(n_iso/gene_isoforms)
        #g_df["cat-subcat_iso_shannon"]=entropy(shannon_iso_cat_subcat)

        #calculate shannon entropy based on reads, supporting cat-subcat combinations
        #shannon_rs_cat_subcat=[]
        #for i in gene[1]["cat_subcat"].unique().tolist():
        #    #get read support for cat-combination
        #    rs=gene[1].loc[gene[1]["cat_subcat"]==i]["isoform_reads"].sum() 
        #    shannon_rs_cat_subcat.append(rs/gene_rs)
        #g_df["cat-subcat_rs_shannon"]=entropy(shannon_rs_cat_subcat)
        
        #shannon_iso_orf=[]
        #calculate shannon entropy based on isoforms, supporting ORFs
        #shanon_iso_cat_orf=[]
        #for i in gene[1]["ORF_seq"].unique().tolist():
        #    n_orf=len(gene[1].loc[gene[1]["ORF_seq"]==i])  
        #    shannon_iso_orf.append(n_orf/gene_isoforms)
        #g_df["orf_iso_shannon"]=entropy(shannon_iso_orf)
        
        acc_df = pd.concat([acc_df, g_df])

        
    # prepare hodgepodge gff
    columns=["chrom","source","feature","hodgepodge_genomic_start_coord","hodgepodge_genomic_end_coord","score","strand","frame","attribute"]
    gff=pd.read_csv(f"/tmp/global2/lvaness/dl20-annex/final/annotation/latest/{acc}_hodgepodgemerged.gff3", sep="\t", header=None, names=columns)
    gff_genes=gff.loc[(gff["feature"]=="gene")|(gff["feature"]=="pseudogene")|(gff["feature"]=="pseudogenic_region")|(gff["feature"]=="transposable_element_gene")]
    gff_genes["hodgepodge_gene"]=gff_genes["attribute"].str.split(";").str[0].str.split("=").str[1]
    gff_genes_sub=gff_genes[["hodgepodge_gene","hodgepodge_genomic_start_coord","hodgepodge_genomic_end_coord"]]

    # add gene coordinates
    acc_df=pd.merge(left=acc_df, right=gff_genes_sub, how='left', left_on='hodgepodge_gene', right_on='hodgepodge_gene')        
        

    # prepare gene type
    nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/nlr_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    nlr["type"]="nlr"
    busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/busco_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    busco["type"]="busco"
    house=pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/housekeeping_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    house["type"]="house"
    all_sets=pd.concat([nlr, busco, house])
    
    custom_dict = {'nlr':0, 'house':1, 'busco':2}  
    all_sets['rank'] = all_sets['type'].map(custom_dict)
    all_sets=all_sets.sort_values(by="rank").drop_duplicates(subset=["hodgepodge_gene"])
    
    # add gene type
    acc_df=pd.merge(left=acc_df, right=all_sets, how='left', left_on='hodgepodge_gene', right_on='hodgepodge_gene')
    acc_df["type"].fillna("other", inplace=True)
    

    col=["chrom","hodgepodge_genomic_start_coord","hodgepodge_genomic_end_coord","strand","hodgepodge_gene","type"]
    for i in acc_df.columns:
        if i not in col:
            col.append(i)
    acc_df=acc_df[col]
    acc_df.fillna(0, inplace=True)
    
    if not os.path.isdir(f"output/08/{acc}"):
        os.makedirs(f"output/08/{acc}")
    
    acc_df["hodgepodge_genomic_start_coord"]=acc_df["hodgepodge_genomic_start_coord"].astype(int)
    acc_df["hodgepodge_genomic_end_coord"]=acc_df["hodgepodge_genomic_end_coord"].astype(int)
    
    acc_df.to_csv(f"output/08/{acc}/{acc}-hodgepodgemerged_shannon_entropies-per-gene.bed", sep="\t", index=False)

In [29]:
if __name__ == '__main__':
    with Pool(17) as p:
        print(p.map(get_entropies_per_gene, accessions))

/tmp/ipykernel_212423/1211782583.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gff_genes["hodgepodge_gene"]=gff_genes["attribute"].str.split(";").str[0].str.split("=").str[1]
/tmp/ipykernel_212423/1211782583.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gff_genes["hodgepodge_gene"]=gff_genes["attribute"].str.split(";").str[0].str.split("=").str[1]
/tmp/ipykernel_212423/1211782583.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[r

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
